# RealMLP at 75,000 training rows

Fits the neural baseline on the signed trial. T4 or L4 is sufficient for an attempt; L4 is preferred for speed and memory headroom.

Run all cells from the top. Setup checks out the selected branch and the training cell opens a fresh Python process so it uses the displayed commit.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_REPO_URL='https://github.com/selimsidan/tfm-credit-benchmark.git'
GITHUB_BRANCH='main'
DRIVE_OUTPUT_DIR='/content/drive/MyDrive/tfm_credit_benchmark'
SOURCE_EXPERIMENT_NAME='lending_club_900k'
TRIAL_EXPERIMENT_NAME='lending_club_75k_trial'
FORCE_RERUN=False
RUN_TESTS=True
# ==================================================================


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess
import sys
import tempfile

drive.mount('/content/drive')
repo=Path(tempfile.mkdtemp(prefix='tfmbench-realmlp-'))/'repo'
subprocess.run(['git','clone','--depth','1','--branch',GITHUB_BRANCH,GITHUB_REPO_URL,str(repo)],check=True)
print('Repository commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip())
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],cwd=repo,check=True)
src_path=str(repo/'src')


In [ ]:
import os

run_env={**os.environ, 'PYTHONPATH':src_path}
if RUN_TESTS:
    subprocess.run([sys.executable,'-m','pytest','-q'],cwd=repo,env=run_env,check=True)


In [ ]:
run_code = """
import json
import sys
import tfmbench
from tfmbench.models import realmlp
from tfmbench.resources import require_cuda
from tfmbench.trial import TrialConfig, run_trial_method

output_root, source_name, trial_name, force_rerun = sys.argv[1:]
print('tfmbench package:', tfmbench.__file__)
print('RealMLP wrapper:', realmlp.__file__)
print(json.dumps(require_cuda('RealMLP 75k trial'), indent=2))
run_trial_method('realmlp', output_root, source_name, trial_name, force_rerun=force_rerun == 'True')
config = TrialConfig(output_root, trial_name, source_name)
print('RealMLP results:', config.experiment_root/'realmlp'/'n75000')
print('Next notebook: 09_tabpfn35_75k.ipynb')
"""
from collections import deque

command = [
    sys.executable, '-u', '-c', run_code, DRIVE_OUTPUT_DIR,
    SOURCE_EXPERIMENT_NAME, TRIAL_EXPERIMENT_NAME, str(FORCE_RERUN),
]
last_lines = deque(maxlen=12)
with subprocess.Popen(
    command, cwd=repo, env=run_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace',
) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
        last_lines.append(line)
    exit_code = process.wait()

if exit_code:
    detail = ''.join(last_lines).strip() or 'The process produced no output.'
    raise RuntimeError(f'RealMLP exited with status {exit_code}. Last output:\n{detail}')
